In [21]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field
from typing_extensions import Literal
from openai import OpenAI
from dotenv import load_dotenv
import os

In [22]:
# Step 1: Setup the Client

load_dotenv(override=True, dotenv_path=".env")
my_api_key = os.getenv("OPENAI_API_KEY")

my_client = OpenAI(api_key=my_api_key)
my_client




In [23]:
# Step 2: Define state structure to track input, decision, and output
class State(TypedDict):
    user_input: str
    decision: str
    output: str

class Route(BaseModel):
    step: Literal["positive", 'negative', 'query'] = Field(None, description="The next step in the routing process")
    

In [24]:
# Step 3: Function to determine the sentiment using AI
from http import client


def get_router_response(user_input: str) -> Route:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "Route the input to 'positive', 'negative', or 'query'. Respond with a JSON object containing the field 'step' with one of these values."
            },
            {
                "role": "user",
                "content": f"Classify the sentiment of the following input: '{user_input}'"
            }
        ],
        temperature=0.7
    )
    classify = response.choices[0].message.content.strip().lower()
    return classify if classify in ["positive", "negative", "query"] else "query" 

In [25]:
# Step 4: Define  functions for each clissification
def handle_positive(state: State) -> State:
    response = client.chat.completions.create(model="gpt-4o-mini",
                    messages=[
                    {"role": "system", "content": "Classify the sentiment based on the input."},
                    {"role": "user", "content": state['input']}],
        max_tokens=500)
    return {"output": response.choices[0].message.content.strip(), "decision": "Positive"}

def handle_negative(state: State) -> State:
    response = client.chat.completions.create(model="gpt-4o-mini",
                    messages=[
                    {"role": "system", "content": "Classify the sentiment based on the input."},
                    {"role": "user", "content": state['input']}],
        max_tokens=500)
    return {"output": response.choices[0].message.content.strip(), "decision": "Negative"}  

def handle_query(state: State) -> State:
    response = client.chat.completions.create(model="gpt-4o-mini",
                    messages=[
                    {"role": "system", "content": "Classify the sentiment based on the input."},
                    {"role": "user", "content": state['input']}],
        max_tokens=500)
    return {"output": response.choices[0].message.content.strip(), "decision": "Query"} 


In [26]:
# Step 5: Routing function to determine which classification function to call
def route_request(state: State) -> str:
    decision = get_router_response(state["input"])
    return {"decision": decision}

def route_decision(state: State) -> str:
    return{
        "positive": handle_positive,
        "negative": handle_negative,
        "query": handle_query       
    }.get(state["decision"], handle_query)


In [27]:
# Step 6: Build LangGraph workflow
def build_workflow() -> StateGraph[State]:
    workflow = StateGraph(State)
    workflow.add_node("handle_positive", handle_positive)
    workflow.add_node("handle_negative", handle_negative)
    workflow.add_node("handle_query", handle_query)
    workflow.add_node("route_request", route_request)

    workflow.add_edge(START, "route_request")
    workflow.add_conditional_edges(
        "route_request",
        route_decision,
        {
            "positive": "handle_positive",
            "negative": "handle_negative",
            "query": "handle_query",
        },
    )
    workflow.add_edge("handle_positive", END)
    workflow.add_edge("handle_negative", END)
    workflow.add_edge("handle_query", END)

    return workflow.compile()


In [28]:
# Step 7: Streamlit Interface
def run_streamlit_app():
    st.title("Sentiment Analysis with LangGraph and OpenAI")
user_input = st.text_area("Enter your text here :")
if st.button("Analyze Sentiment"):
    initial_state: State = {
        "user_input": user_input,
        "decision": "",
        "output": ""
    }
    workflow = build_workflow()
    final_state = workflow.run(initial_state)
    st.write(f"**Sentiment Decision:** {final_state['decision']}")
    st.write(f"**Output:** {final_state['output']}")

if __name__ == "__main__":
    run_streamlit_app()

2026-01-13 16:21:49.837 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-13 16:21:49.838 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-13 16:21:49.838 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-13 16:21:49.839 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-13 16:21:49.840 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-13 16:21:49.841 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-13 16:21:49.841 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-13 16:21:49.842 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar